In [ ]:
import sys, os
sys.path.append('./src/')

from VAE_variants import VAE, CVAE, CSVAENA, CSVAE, HCSVAENA, HCSVAE, DLVAE, SDIVA, CCVAE
from VAE_trainers import EpochPyroTrainer, AdversarialEpochPyroTrainer, ThresholdPyroTrainer, AdversarialThresholdPyroTrainer
from matplotlib.colors import LinearSegmentedColormap
from sklearn.naive_bayes import GaussianNB
from metrics import MINE
from tqdm import trange
import pyro.distributions as dist


import torch, pyro
import numpy as np
import matplotlib.pyplot as plt
import pyro.optim as opt
import seaborn as sns

np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

demo_epochs=5

cmap = LinearSegmentedColormap.from_list("cmap", ["#DB6D00", "#006DDB"])

## Data

In [ ]:
import torch.utils.data as utils

#################### DATASET PARAMS #########################################################################################
n=10000  # Number of points for the roll
batch_size=64 # Loader batch size
p=0.3
##################################################################################################################################
from sklearn.datasets import make_swiss_roll

np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

xs, _ = make_swiss_roll(n)
xs = torch.FloatTensor(xs)
ys = (xs[:, 1] < 10).type(torch.long)

ys_real = ys.float().reshape(-1,1) 

for i in range(len(ys)):
    if np.random.uniform() < p:
        ys[i] = abs(ys[i]-1)

ys = ys.float().reshape(-1,1)
xs = (xs - np.array([xs[:,0].min() - 5, -5, xs[:,2].min() - 5])).float() # Shift to make entries non-negative
dataset = utils.TensorDataset(xs, ys, ys) # Duplicate ys, needed for workflow
train_set = dataset
train_loader = torch.utils.data.DataLoader(train_set, shuffle=True, batch_size=batch_size)


xs, _ = make_swiss_roll(n)
xs = torch.FloatTensor(xs)
ys = (xs[:, 1] < 10).type(torch.long)

for i in range(len(ys)):
    if np.random.uniform() < p:
        ys[i] = abs(ys[i]-1)

ys = ys.float().reshape(-1,1)
xs = (xs - np.array([xs[:,0].min() - 5, -5, xs[:,2].min() - 5])).float() # Shift to make entries non-negative
dataset_test = utils.TensorDataset(xs, ys, ys) # Duplicate ys, needed for workflow
test_set = dataset_test
test_loader = torch.utils.data.DataLoader(test_set, shuffle=False, batch_size=batch_size)

In [ ]:
test_set_marg = test_set[:][0].clone()
test_set_marg[:, 1] = test_set_marg[:, 1].mean()

In [ ]:
fig = plt.figure(figsize=(15,15))
ax = fig.add_subplot(projection='3d')
scatter = ax.scatter(dataset[:][0][:,0], dataset[:][0][:,1], dataset[:][0][:,2], c=train_set[:][1].numpy(), cmap=cmap, alpha=0.6)
"""
elems = list(scatter.legend_elements())
legend = ax.legend(*elems,
               loc="lower left", 
               title="Classes",
               title_fontsize=20,
               fontsize=18,
                markerscale=3)
"""

ax.axis('off')

ax.set_xlabel('X', weight='bold', labelpad=30, fontsize=18)
ax.set_ylabel('Y', weight='bold', labelpad=30, fontsize=18)
ax.set_zlabel('Z', weight='bold', labelpad=7, fontsize=18)

ax.set_xlim(0,30)
ax.set_ylim(0,30)
ax.set_zlim(0,30)
plt.show()

In [ ]:
fig = plt.figure(figsize=(15,15))
ax = fig.add_subplot(projection='3d')
scatter = ax.scatter(dataset[:][0][:,0], torch.FloatTensor([dataset[:][0][:,1].mean().item()]*len(dataset)).reshape(-1,1), dataset[:][0][:,2], c=train_set[:][1].numpy(), cmap=cmap, alpha=0.6)
"""
elems = list(scatter.legend_elements())
legend = ax.legend(*elems,
               loc="lower left", 
               title="Classes",
               title_fontsize=20,
               fontsize=18,
                markerscale=3)
"""

ax.axis('off')

ax.set_xlabel('X', weight='bold', labelpad=30, fontsize=18)
ax.set_ylabel('Y', weight='bold', labelpad=30, fontsize=18)
ax.set_zlabel('Z', weight='bold', labelpad=7, fontsize=18)

ax.set_xlim(0,30)
ax.set_ylim(0,30)
ax.set_zlim(0,30)
plt.show()

In [ ]:
plt.scatter(dataset[:][0][:,0], dataset[:][0][:,2], c=train_set[:][1].numpy(), cmap=cmap, alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
sns.stripplot(dataset[:][0][:,1], c=train_set[:][1].numpy(), cmap=cmap, alpha=0.6, orient='y')
plt.axis('off')
plt.show()

In [ ]:
orig_data_score = GaussianNB().fit(train_set[:][0], train_set[:][1].squeeze(-1)).score(test_set[:][0], test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc - Combined: {np.round(orig_data_score, 2)}')

In [ ]:
orig_data_score_common = GaussianNB(priors=(0.49, 0.51)).fit(train_set[:][0][:,[0,2]], train_set[:][1].squeeze(-1)).score(test_set[:][0][:,[0,2]], test_set[:][1].squeeze(-1)) # sample more points if needed
print(f'Bayes classifier acc - Common: {np.round(orig_data_score_common, 2)}')

In [ ]:
orig_data_score_cond = GaussianNB().fit(train_set[:][0][:, 1].reshape(-1,1), train_set[:][1].squeeze(-1)).score(test_set[:][0][:, 1].reshape(-1,1), test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc - Cond: {np.round(orig_data_score_cond, 2)}')

## CSVAE - No Adv.

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

csvaena = CSVAENA(3, [1], latent_dim=2, w_dim=2, num_layers=2, recon_weight=20, z_kl_weight=2e-1)
csvaena_trainer = EpochPyroTrainer(demo_epochs, csvaena, train_loader, test_loader)
csvaena_trainer.train()

In [ ]:
preds = csvaena_trainer.get_variables('test')
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
recons = preds['rec'][0, 0].cpu()

In [ ]:
mi_z_w = MINE(4, [128]).cuda().mutual_information(z_s.cuda(), w_s.cuda())
print(f'I(z ; w) = {np.round(mi_z_w, 3)}')

In [ ]:
plt.scatter(z_s[:,0], z_s[:,1], c=test_set[:][1].numpy(), cmap=cmap, s=10, alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(w_s[:,0], w_s[:,1], c=test_set[:][1].numpy(), cmap=cmap, s=10, alpha=0.6)
plt.axis('off')
plt.gca().set_box_aspect(1)

plt.show()

In [ ]:
fig = plt.figure(figsize=(15,15))
ax = fig.add_subplot(projection='3d')
scatter = ax.scatter(recons[:,0], recons[:,1], recons[:,2], c=train_set[:][1].numpy(), cmap=cmap)

ax.grid(False)

ax.set_xlabel('X', weight='bold', labelpad=30, fontsize=18)
ax.set_ylabel('Y', weight='bold', labelpad=30, fontsize=18)
ax.set_zlabel('Z', weight='bold', labelpad=7, fontsize=18)

ax.set_xlim(0,30)
ax.set_ylim(0,30)
ax.set_zlim(0,30)
ax.axis('off')

plt.show()

In [ ]:
print((test_set[:][0] - recons).square().mean().sqrt())
print(-1 * dist.Normal(recons.mean(dim=0), recons.std(dim=0)).log_prob(test_set[:][0]).sum(dim=1).mean())

In [ ]:
combined = torch.concatenate((z_s, w_s), dim=1)

assert combined.shape[0] == test_set[:][0].shape[0]
score = GaussianNB().fit(combined, test_set[:][1].squeeze(-1)).score(combined, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 4)}')
print(f'Diff: {np.round(abs(np.round(orig_data_score, 2) - np.round(score, 4)), 4)}' )

In [ ]:
score = GaussianNB().fit(z_s, test_set[:][1].squeeze(-1)).score(z_s, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 4)}')
print(f'Diff: {np.round(abs(np.round(orig_data_score_common, 2) - np.round(score, 4)), 4)}' )

In [ ]:
score = GaussianNB().fit(w_s, test_set[:][1].squeeze(-1)).score(w_s, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 4)}')
print(f'Diff: {np.round(abs(np.round(orig_data_score_cond, 2) - np.round(score, 4)), 4)}' )

## CSVAE

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

csvae = CSVAE(3, [1], latent_dim=2, w_dim=2, num_layers=2, recon_weight=20, adversarial_weight=50, z_kl_weight=2e-1)  
csvae_trainer = AdversarialEpochPyroTrainer(demo_epochs, 1, 1, csvae, train_loader, test_loader)
csvae_trainer.train()

In [ ]:
preds = csvae_trainer.get_variables('test')
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
recons = preds['rec'][0, 0].cpu()

In [ ]:
mi_z_w = MINE(4, [128]).cuda().mutual_information(z_s.cuda(), w_s.cuda())
print(f'I(z ; w) = {np.round(mi_z_w, 3)}')

In [ ]:
print((test_set[:][0] - recons).square().mean().sqrt())
print(-1 * dist.Normal(recons.mean(dim=0), recons.std(dim=0)).log_prob(test_set[:][0]).sum(dim=1).mean())

In [ ]:
plt.scatter(z_s[:,0], z_s[:,1], c=test_set[:][1].numpy(), cmap=cmap, s=10, alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(w_s[:,0], w_s[:,1], c=test_set[:][1].numpy(), cmap=cmap, s=10, alpha=0.6)
plt.axis('off')
plt.gca().set_box_aspect(1)

plt.show()

In [ ]:
fig = plt.figure(figsize=(15,15))
ax = fig.add_subplot(projection='3d')
scatter = ax.scatter(recons[:,0], recons[:,1], recons[:,2], c=test_set[:][1].numpy(), cmap=cmap)

ax.grid(False)

ax.set_xlabel('X', weight='bold', labelpad=30, fontsize=18)
ax.set_ylabel('Y', weight='bold', labelpad=30, fontsize=18)
ax.set_zlabel('Z', weight='bold', labelpad=7, fontsize=18)

ax.set_xlim(0,30)
ax.set_ylim(0,30)

ax.axis('off')



ax.set_zlim(0,30)
plt.show()

In [ ]:
combined = torch.concatenate((z_s, w_s), dim=1)

assert combined.shape[0] == test_set[:][0].shape[0]
score = GaussianNB().fit(combined, test_set[:][1].squeeze(-1)).score(combined, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 4)}')
print(f'Diff: {np.round(abs(np.round(orig_data_score, 2) - np.round(score, 4)), 4)}' )

In [ ]:
score = GaussianNB().fit(z_s, test_set[:][1].squeeze(-1)).score(z_s, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 4)}')
print(f'Diff: {np.round(abs(np.round(orig_data_score_common, 2) - np.round(score, 4)), 4)}' )

In [ ]:
score = GaussianNB().fit(w_s, test_set[:][1].squeeze(-1)).score(w_s, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 4)}')
print(f'Diff: {np.round(abs(np.round(orig_data_score_cond, 2) - np.round(score, 4)), 4)}' )

## HCSVAE - No Adv.

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

hcsvaena = HCSVAENA(3, [1], latent_dim=2, w_dim=2, num_layers=2, recon_weight=20, z_kl_weight=2e-1)
hcsvaena_trainer = EpochPyroTrainer(demo_epochs, hcsvaena, train_loader, test_loader)
hcsvaena_trainer.train()

In [ ]:
preds = hcsvaena_trainer.get_variables('test')
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
recons = preds['rec'][0, 0].cpu()

In [ ]:
mi_z_w = MINE(4, [128]).cuda().mutual_information(z_s.cuda(), w_s.cuda())
print(f'I(z ; w) = {np.round(mi_z_w, 3)}')

In [ ]:
print((test_set[:][0] - recons).square().mean().sqrt())
print(-1 * dist.Normal(recons.mean(dim=0), recons.std(dim=0)).log_prob(test_set[:][0]).sum(dim=1).mean())

In [ ]:
plt.scatter(z_s[:,0], z_s[:,1], c=test_set[:][1].numpy(), cmap=cmap, s=10, alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(w_s[:,0], w_s[:,1], c=test_set[:][1].numpy(), cmap=cmap, s=10, alpha=0.6)
plt.axis('off')

plt.gca().set_box_aspect(1)

plt.show()

In [ ]:
fig = plt.figure(figsize=(15,15))
ax = fig.add_subplot(projection='3d')
scatter = ax.scatter(recons[:,0], recons[:,1], recons[:,2], c=test_set[:][1].numpy(), cmap=cmap)

ax.grid(False)

ax.set_xlabel('X', weight='bold', labelpad=30, fontsize=18)
ax.set_ylabel('Y', weight='bold', labelpad=30, fontsize=18)
ax.set_zlabel('Z', weight='bold', labelpad=7, fontsize=18)

ax.axis('off')

ax.set_xlim(0,30)
ax.set_ylim(0,30)
ax.set_zlim(0,30)

plt.show()

In [ ]:
combined = torch.concatenate((z_s, w_s), dim=1)

assert combined.shape[0] == test_set[:][0].shape[0]
score = GaussianNB().fit(combined, test_set[:][1].squeeze(-1)).score(combined, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 4)}')
print(f'Diff: {np.round(abs(np.round(orig_data_score, 2) - np.round(score, 4)), 4)}' )

In [ ]:
score = GaussianNB().fit(z_s, test_set[:][1].squeeze(-1)).score(z_s, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 4)}')
print(f'Diff: {np.round(abs(np.round(orig_data_score_common, 2) - np.round(score, 4)), 4)}' )

In [ ]:
score = GaussianNB().fit(w_s, test_set[:][1].squeeze(-1)).score(w_s, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 4)}')
print(f'Diff: {np.round(abs(np.round(orig_data_score_cond, 2) - np.round(score, 4)), 4)}' )

## DIVA

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

diva = SDIVA(3, [1], latent_dim=2, w_dim=2, num_layers=2, recon_weight=20, kl_weight=2e-1)
diva_trainer = EpochPyroTrainer(demo_epochs, diva, train_loader, test_loader)
diva_trainer.train()

In [ ]:
preds = diva_trainer.get_variables('test')
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
recons = preds['rec'][0, 0].cpu()

In [ ]:
mi_z_w = MINE(4, [128]).cuda().mutual_information(z_s.cuda(), w_s.cuda())
print(f'I(z ; w) = {np.round(mi_z_w, 3)}')

In [ ]:
print((test_set[:][0] - recons).square().mean().sqrt())
print(-1 * dist.Normal(recons.mean(dim=0), recons.std(dim=0)).log_prob(test_set[:][0]).sum(dim=1).mean())

In [ ]:
plt.scatter(z_s[:,0], z_s[:,1], c=test_set[:][1].numpy(), cmap=cmap, s=10, alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(w_s[:,0], w_s[:,1], c=test_set[:][1].numpy(), cmap=cmap, s=10, alpha=0.6)
plt.axis('off')

plt.gca().set_box_aspect(1)

plt.show()

In [ ]:
fig = plt.figure(figsize=(15,15))
ax = fig.add_subplot(projection='3d')
scatter = ax.scatter(recons[:,0], recons[:,1], recons[:,2], c=test_set[:][1].numpy(), cmap=cmap)


ax.set_xlabel('X', weight='bold', labelpad=30, fontsize=18)
ax.set_ylabel('Y', weight='bold', labelpad=30, fontsize=18)
ax.set_zlabel('Z', weight='bold', labelpad=7, fontsize=18)

ax.set_xlim(0,30)
ax.set_ylim(0,30)
ax.set_zlim(0,30)

ax.axis('off')


plt.show()

In [ ]:
combined = torch.concatenate((z_s, w_s), dim=1)

assert combined.shape[0] == test_set[:][0].shape[0]
score = GaussianNB().fit(combined, test_set[:][1].squeeze(-1)).score(combined, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 4)}')
print(f'Diff: {np.round(abs(np.round(orig_data_score, 2) - np.round(score, 4)), 4)}' )

In [ ]:
score = GaussianNB().fit(z_s, test_set[:][1].squeeze(-1)).score(z_s, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 4)}')
print(f'Diff: {np.round(abs(np.round(orig_data_score_common, 2) - np.round(score, 4)), 4)}' )

In [ ]:
score = GaussianNB().fit(w_s, test_set[:][1].squeeze(-1)).score(w_s, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 4)}')
print(f'Diff: {np.round(abs(np.round(orig_data_score_cond, 2) - np.round(score, 4)), 4)}' )

## CCVAE

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

ccvae = CCVAE(3, [1], latent_dim=2, w_dim=2, num_layers=2, recon_weight=20, kl_weight=2e-1)
ccvae_trainer = EpochPyroTrainer(demo_epochs, ccvae, train_loader, test_loader)
ccvae_trainer.train()

In [ ]:
preds = ccvae_trainer.get_variables('test')
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
recons = preds['rec'][0, 0].cpu()

In [ ]:
mi_z_w = MINE(4, [128]).cuda().mutual_information(z_s.cuda(), w_s.cuda())
print(f'I(z ; w) = {np.round(mi_z_w, 3)}')

In [ ]:
print((test_set[:][0] - recons).square().mean().sqrt())
print(-1 * dist.Normal(recons.mean(dim=0), recons.std(dim=0)).log_prob(test_set[:][0]).sum(dim=1).mean())

In [ ]:
plt.scatter(z_s[:,0], z_s[:,1], c=test_set[:][1].numpy(), cmap=cmap, s=10, alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(w_s[:,0], w_s[:,1], c=test_set[:][1].numpy(), cmap=cmap, s=10, alpha=0.6)
plt.axis('off')

plt.gca().set_box_aspect(1)

plt.show()

In [ ]:
fig = plt.figure(figsize=(15,15))
ax = fig.add_subplot(projection='3d')
scatter = ax.scatter(recons[:,0], recons[:,1], recons[:,2], c=test_set[:][1].numpy(), cmap=cmap)

ax.grid(False)

ax.set_xlabel('X', weight='bold', labelpad=30, fontsize=18)
ax.set_ylabel('Y', weight='bold', labelpad=30, fontsize=18)
ax.set_zlabel('Z', weight='bold', labelpad=7, fontsize=18)

ax.set_xlim(0,30)
ax.set_ylim(0,30)
ax.set_zlim(0,30)

ax.axis('off')

plt.show()

In [ ]:
combined = torch.concatenate((z_s, w_s), dim=1)

assert combined.shape[0] == test_set[:][0].shape[0]
score = GaussianNB().fit(combined, test_set[:][1].squeeze(-1)).score(combined, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 4)}')
print(f'Diff: {np.round(abs(np.round(orig_data_score, 2) - np.round(score, 4)), 4)}' )

In [ ]:
score = GaussianNB().fit(z_s, test_set[:][1].squeeze(-1)).score(z_s, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 4)}')
print(f'Diff: {np.round(abs(np.round(orig_data_score_common, 2) - np.round(score, 4)), 4)}' )

In [ ]:
score = GaussianNB().fit(w_s, test_set[:][1].squeeze(-1)).score(w_s, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 4)}')
print(f'Diff: {np.round(abs(np.round(orig_data_score_cond, 2) - np.round(score, 4)), 4)}' )

## HCSVAE

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

hcsvae = HCSVAE(3, [1], latent_dim=2, w_dim=2, num_layers=2, recon_weight=20, adversarial_weight=50, z_kl_weight=5e-1)
hcsvae_trainer = AdversarialEpochPyroTrainer(demo_epochs, 1, 1, hcsvae, train_loader, test_loader)
hcsvae_trainer.train()

In [ ]:
preds = hcsvae_trainer.get_variables('test')
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
recons = preds['rec'][0, 0].cpu()

In [ ]:
mi_z_w = MINE(4, [128]).cuda().mutual_information(z_s.cuda(), w_s.cuda())
print(f'I(z ; w) = {np.round(mi_z_w, 3)}')

In [ ]:
print((test_set[:][0] - recons).square().mean().sqrt())
print(-1 * dist.Normal(recons.mean(dim=0), recons.std(dim=0)).log_prob(test_set[:][0]).sum(dim=1).mean())

In [ ]:
plt.scatter(z_s[:,0], z_s[:,1], c=test_set[:][1].numpy(), cmap=cmap, s=10, alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(w_s[:,0], w_s[:,1], c=test_set[:][1].numpy(), cmap=cmap, s=10, alpha=0.6)
plt.axis('off')

plt.gca().set_box_aspect(1)
plt.show()

In [ ]:
fig = plt.figure(figsize=(15,15))
ax = fig.add_subplot(projection='3d')
scatter = ax.scatter(recons[:,0], recons[:,1], recons[:,2], c=test_set[:][1].numpy(), cmap=cmap)

ax.grid(False)

ax.set_xlabel('X', weight='bold', labelpad=30, fontsize=18)
ax.set_ylabel('Y', weight='bold', labelpad=30, fontsize=18)
ax.set_zlabel('Z', weight='bold', labelpad=7, fontsize=18)

ax.set_xlim(0,30)
ax.set_ylim(0,30)
ax.set_zlim(0,30)

ax.axis('off')
plt.show()

In [ ]:
combined = torch.concatenate((z_s, w_s), dim=1)

assert combined.shape[0] == test_set[:][0].shape[0]
score = GaussianNB().fit(combined, test_set[:][1].squeeze(-1)).score(combined, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 4)}')
print(f'Diff: {np.round(abs(np.round(orig_data_score, 2) - np.round(score, 4)), 4)}' )

In [ ]:
score = GaussianNB().fit(z_s, test_set[:][1].squeeze(-1)).score(z_s, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 4)}')
print(f'Diff: {np.round(abs(np.round(orig_data_score_common, 2) - np.round(score, 4)), 4)}' )

In [ ]:
score = GaussianNB().fit(w_s, test_set[:][1].squeeze(-1)).score(w_s, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 4)}')
print(f'Diff: {np.round(abs(np.round(orig_data_score_cond, 2) - np.round(score, 4)), 4)}' )

## DISCoVeR (Ours)

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
pyro.util.set_rng_seed(42)

dlvae = DLVAE(3, [1], latent_dim=2, w_dim=2, num_layers=2, recon_weight=9e-1, recon_weight_z=1e-1, z_kl_weight=2e-1, w_kl_weight=2e-1, adversarial_weight=8)
dlvae_trainer = AdversarialEpochPyroTrainer(demo_epochs, 1, 1, dlvae, train_loader, test_loader)
dlvae_trainer.train()

In [ ]:
preds = dlvae_trainer.get_variables('test')
z_s = preds['z'][0].cpu()
w_s = preds['w'][0].cpu()
recons_z = preds['rec_z'][0, 0].cpu()
recons_w = preds['rec_w'][0, 0].cpu()

In [ ]:
mi_z_w = MINE(4, [128]).cuda().mutual_information(z_s.cuda(), w_s.cuda(), steps=100)
print(f'I(z ; w) = {np.round(mi_z_w, 3)}')

In [ ]:
print((test_set[:][0] - recons_w).square().mean().sqrt())
print(-1 * dist.Normal(recons_w.mean(dim=0), recons_w.std(dim=0)).log_prob(test_set[:][0]).sum(dim=1).mean())

In [ ]:
plt.scatter(z_s[:,0], z_s[:,1], c=test_set[:][1].numpy(), cmap=cmap, s=10, alpha=0.6)
plt.axis('off')
plt.show()

In [ ]:
plt.scatter(w_s[:,0], w_s[:,1], c=test_set[:][1].numpy(), cmap=cmap.reversed(), s=10, alpha=0.6)
plt.axis('off')
plt.gca().set_box_aspect(1)

plt.show()

In [ ]:
fig = plt.figure(figsize=(15,15))
ax = fig.add_subplot(projection='3d')
scatter = ax.scatter(recons_z[:,0], recons_z[:,1], recons_z[:,2], c=test_set[:][1].numpy(), cmap=cmap)

ax.axis('off')

ax.set_xlabel('X', weight='bold', labelpad=30, fontsize=18)
ax.set_ylabel('Y', weight='bold', labelpad=30, fontsize=18)
ax.set_zlabel('Z', weight='bold', labelpad=7, fontsize=18)

ax.set_xlim(0,30)
ax.set_ylim(0,30)
ax.set_zlim(0,30)


plt.show()

In [ ]:
fig = plt.figure(figsize=(15,15))
ax = fig.add_subplot(projection='3d')
scatter = ax.scatter(recons_w[:,0], recons_w[:,1], recons_w[:,2], c=test_set[:][1].numpy(), cmap=cmap)

ax.grid(False)
ax.axis('off')

ax.set_xlabel('X', weight='bold', labelpad=30, fontsize=18)
ax.set_ylabel('Y', weight='bold', labelpad=30, fontsize=18)
ax.set_zlabel('Z', weight='bold', labelpad=7, fontsize=18)

ax.set_xlim(0,30)
ax.set_ylim(0,30)
ax.set_zlim(0,30)
ax.axis('off')

plt.show()

In [ ]:
combined = torch.concatenate((z_s, w_s), dim=1)

assert combined.shape[0] == test_set[:][0].shape[0]
score = GaussianNB().fit(combined, test_set[:][1].squeeze(-1)).score(combined, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 4)}')
print(f'Diff: {np.round(abs(np.round(orig_data_score, 2) - np.round(score, 4)), 4)}' )

In [ ]:
score = GaussianNB().fit(z_s, test_set[:][1].squeeze(-1)).score(z_s, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 4)}')
print(f'Diff: {np.round(abs(np.round(orig_data_score_common, 2) - np.round(score, 4)), 4)}' )

In [ ]:
score = GaussianNB().fit(w_s, test_set[:][1].squeeze(-1)).score(w_s, test_set[:][1].squeeze(-1))
print(f'Bayes classifier acc: {np.round(score, 4)}')
print(f'Diff: {np.round(abs(np.round(orig_data_score_cond, 2) - np.round(score, 4)), 4)}' )